In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import plotly.graph_objects as go

df = pd.read_csv(
    "../data/processed/macro_data_processed.csv",
    index_col="date",
    parse_dates=True
)

# calcula inflação anual e misery index
df["inflation"] = df["cpi"].pct_change(12) * 100
df["misery_index"] = df["unemployment"] + df["inflation"]

misery = df[["unemployment", "inflation", "misery_index", "recession"]].dropna()

print(misery.shape)
misery.describe()

(661, 4)


C:\Users\Usuario\AppData\Local\Temp\ipykernel_13936\728448924.py:15: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df["inflation"] = df["cpi"].pct_change(12) * 100


,unemployment,inflation,misery_index,recession
count,661.000000,661.000000,661.000000,661.000000
mean,6.073525,3.958231,10.031756,0.111952
std,1.721542,2.905860,3.475695,0.315546
min,3.400000,-1.958761,5.008843,0.000000
25%,4.800000,2.144608,7.451287,0.000000
50%,5.700000,3.120464,9.266332,0.000000
75%,7.200000,4.741834,11.619242,0.000000
max,14.800000,14.592275,21.925770,1.000000


In [2]:
rec = misery["recession"]
starts = rec[(rec == 1) & (rec.shift(1) == 0)].index.tolist()
ends = rec[(rec == 0) & (rec.shift(1) == 1)].index.tolist()
if rec.iloc[0] == 1:
    starts = [rec.index[0]] + starts
pairs = list(zip(starts, ends[:len(starts)]))

fig = go.Figure()

# componentes empilhados
fig.add_trace(go.Scatter(
    x=misery.index, y=misery["unemployment"],
    mode="lines", name="Unemployment",
    line=dict(color="#5cb8e0", width=1.2),
    stackgroup="one",
    fillcolor="rgba(92, 184, 224, 0.4)",
))

fig.add_trace(go.Scatter(
    x=misery.index, y=misery["inflation"],
    mode="lines", name="Inflation",
    line=dict(color="#e05c5c", width=1.2),
    stackgroup="one",
    fillcolor="rgba(224, 92, 92, 0.4)",
))

# linha do misery index
fig.add_trace(go.Scatter(
    x=misery.index, y=misery["misery_index"],
    mode="lines", name="Misery Index",
    line=dict(color="white", width=2),
))

# média histórica
fig.add_hline(
    y=misery["misery_index"].mean(),
    line_dash="dash", line_color="yellow",
    opacity=0.6,
    annotation_text=f"Historical avg: {misery['misery_index'].mean():.1f}",
    annotation_position="top left",
    annotation_font_color="yellow",
)

# recessões
for start, end in pairs:
    fig.add_vrect(
        x0=start, x1=end,
        fillcolor="gray", opacity=0.2,
        layer="below", line_width=0,
    )

fig.update_layout(
    title_text="US Misery Index (1970–2026) — Unemployment + Inflation",
    title_font_size=14,
    xaxis_title="Date",
    yaxis_title="Misery Index",
    paper_bgcolor="#111111",
    plot_bgcolor="#111111",
    font=dict(color="white"),
    height=500,
    legend=dict(orientation="h", y=1.08),
)

fig.update_xaxes(gridcolor="#333333")
fig.update_yaxes(gridcolor="#333333")

fig.show()
fig.write_html("../outputs/figures/04_misery_index.html")